# Arquitetura de todo o projeto

# Arquitetura de Dados — Data Lake / Lakehouse

## 1. Objetivo

Este documento descreve uma estrutura base de organização de dados em um Data Lake/Lakehouse, utilizando uma arquitetura em camadas inspirada no modelo medalhão: **bronze**, **silver** e **gold**.

A proposta é separar os dados por projeto, estágio de tratamento e finalidade de uso, facilitando governança, rastreabilidade, manutenção dos pipelines de ETL/ELT e consumo analítico em ferramentas como Power BI, Databricks, Fabric ou outros ambientes de dados.

---

## 2. Estrutura geral

A estrutura principal parte de um **storage account** e de um **container**, onde cada projeto possui sua própria pasta raiz.

```text
<storage-account>/<container>/
│
├── projeto1/
│   │
│   ├── resource/                         # dados de origem e entrega
│   │   ├── origem/                       # arquivos CSV/Parquet recebidos da fonte
│   │   └── destino/                      # exportações para clientes ou setores
│   │
│   ├── bronze/                           # dados crus / raw
│   │   ├── clientes/
│   │   ├── pedidos/
│   │   ├── produtos/
│   │   └── categorias/
│   │
│   ├── silver/                           # dados limpos / tratados
│   │   ├── clientes/
│   │   ├── pedidos/
│   │   └── produtos/
│   │
│   ├── gold/                             # dados agregados / analytics
│   │   ├── vendas_categoria/
│   │   └── vendas_cidade_MG/
│   │
│   ├── metadata/                         # metadados e logs
│   │   ├── bronze/
│   │   ├── silver/
│   │   ├── gold/
│   │   ├── ddl/                          # scripts de criação de tabelas
│   │   ├── logs/                         # logs de execução do ETL
│   │   └── checkpoints/                  # checkpoints do Autoloader / streaming
│   │
│   └── tmp/                              # staging temporário
│
└── projeto2/
```

> Observação: esta é uma estrutura básica de exemplo. Cada empresa, setor ou projeto pode adaptar os nomes e níveis conforme a necessidade, mantendo a separação lógica entre origem, tratamento, consumo e controle operacional.

---

## 3. Descrição das camadas

### 3.1 `resource/`

A pasta `resource` concentra arquivos de entrada e saída que não necessariamente representam tabelas tratadas do lakehouse.

| Subpasta | Finalidade |
|---|---|
| `resource/origem/` | Armazenar arquivos recebidos da fonte original, como CSV, Excel, JSON ou Parquet. |
| `resource/destino/` | Armazenar exportações destinadas a clientes, áreas internas ou outros sistemas. |

Exemplos de uso:

```text
resource/origem/clientes_2026_05_01.csv
resource/origem/pedidos_2026_05_01.parquet
resource/destino/vendas_consolidadas_cliente.xlsx
```

---

### 3.2 `bronze/`

A camada **bronze** armazena os dados crus, com o mínimo possível de transformação.

Principais características:

- mantém os dados próximos do formato original;
- preserva histórico de carga;
- serve como ponto de auditoria e reprocessamento;
- pode conter colunas técnicas, como `dt_ingestao`, `arquivo_origem`, `hash_linha` e `id_carga`.

Exemplo:

```text
bronze/clientes/
bronze/pedidos/
bronze/produtos/
bronze/categorias/
```

---

### 3.3 `silver/`

A camada **silver** armazena dados tratados, padronizados e prontos para relacionamento entre entidades.

Principais tratamentos esperados:

- padronização de nomes de colunas;
- ajuste de tipos de dados;
- tratamento de nulos;
- remoção ou controle de duplicidades;
- aplicação de regras de qualidade;
- criação de chaves técnicas ou chaves de relacionamento.

Exemplo:

```text
silver/clientes/
silver/pedidos/
silver/produtos/
```

---

### 3.4 `gold/`

A camada **gold** armazena dados agregados, modelados ou enriquecidos para consumo analítico.

Normalmente é a camada utilizada por dashboards, relatórios, APIs analíticas e áreas de negócio.

Exemplo:

```text
gold/vendas_categoria/
gold/vendas_cidade_MG/
```

Possíveis tabelas gold:

| Tabela | Finalidade |
|---|---|
| `vendas_categoria` | Consolidar vendas por categoria de produto. |
| `vendas_cidade_MG` | Consolidar vendas por cidade do estado de Minas Gerais. |

---

## 4. Metadados, logs e controle operacional

A pasta `metadata/` centraliza arquivos e informações de controle da arquitetura.

```text
metadata/
├── bronze/
├── silver/
├── gold/
├── ddl/
├── logs/
└── checkpoints/
```

| Subpasta | Finalidade |
|---|---|
| `metadata/bronze/` | Metadados específicos das tabelas ou cargas da camada bronze. |
| `metadata/silver/` | Metadados específicos das tabelas ou cargas da camada silver. |
| `metadata/gold/` | Metadados específicos das tabelas ou cargas da camada gold. |
| `metadata/ddl/` | Scripts de criação e alteração de tabelas. |
| `metadata/logs/` | Logs de execução dos pipelines de ETL/ELT. |
| `metadata/checkpoints/` | Checkpoints utilizados por Autoloader, streaming ou cargas incrementais. |

---

## 5. Staging temporário

A pasta `tmp/` deve ser usada apenas para dados temporários de processamento.

Exemplos de uso:

- arquivos intermediários;
- resultados parciais de uma carga;
- validações temporárias;
- dados que serão descartados após o processamento.

Recomendação: definir política de retenção para evitar acúmulo desnecessário de arquivos temporários.

---

## 6. Fluxo recomendado dos dados

```text
Fonte de dados
     │
     ▼
resource/origem
     │
     ▼
bronze
     │
     ▼
silver
     │
     ▼
gold
     │
     ├── Power BI
     ├── Databricks SQL
     ├── APIs analíticas
     └── Exportações em resource/destino
```

Resumo do fluxo:

1. Os arquivos são recebidos em `resource/origem/`.
2. A ingestão grava os dados crus em `bronze/`.
3. As regras de limpeza e padronização geram dados em `silver/`.
4. As regras de negócio e agregações geram tabelas em `gold/`.
5. Os dados finais podem ser consumidos por dashboards ou exportados para `resource/destino/`.

---

## 7. Boas práticas de organização

### 7.1 Padronização de nomes

Recomenda-se utilizar nomes em letras minúsculas, sem espaços e com separador `_`.

Exemplos:

```text
clientes
pedidos
produtos
vendas_categoria
vendas_cidade_mg
```

Evitar:

```text
Clientes Finais
Vendas por Categoria
Tabela Nova 1
```

---

### 7.2 Particionamento

Para tabelas grandes, é recomendado particionar os dados por campos de data ou período.

Exemplo:

```text
bronze/pedidos/ano=2026/mes=05/dia=01/
silver/pedidos/ano=2026/mes=05/
gold/vendas_categoria/ano=2026/mes=05/
```

Campos comuns para particionamento:

- `ano`
- `mes`
- `dia`
- `dt_ingestao`
- `dt_referencia`

---

### 7.3 Colunas técnicas recomendadas

| Coluna | Finalidade |
|---|---|
| `dt_ingestao` | Data e hora em que o dado foi carregado. |
| `arquivo_origem` | Nome ou caminho do arquivo original. |
| `id_carga` | Identificador único da execução da carga. |
| `hash_linha` | Hash para controle de duplicidade ou alteração. |
| `dt_atualizacao` | Data de atualização do registro. |

---

## 8. Consumo analítico

A camada recomendada para consumo no Power BI é a **gold**, pois ela já deve conter dados tratados, regras de negócio aplicadas e agregações necessárias para análise.

Quando houver necessidade de análise detalhada ou investigação técnica, a camada **silver** também pode ser utilizada, desde que os dados estejam padronizados e com qualidade adequada.

Evite conectar dashboards diretamente na camada **bronze**, pois ela representa dados crus e pode conter inconsistências, duplicidades ou formatos ainda não tratados.

---

## 9. Resumo das responsabilidades por camada

| Camada | Tipo de dado | Consumidor principal | Finalidade |
|---|---|---|---|
| `resource/origem` | Arquivos recebidos | Engenharia de Dados | Entrada de dados da fonte. |
| `bronze` | Dados crus | Engenharia de Dados | Preservação e reprocessamento. |
| `silver` | Dados tratados | Engenharia e Analytics | Dados limpos e relacionáveis. |
| `gold` | Dados agregados | Negócio / BI | Dashboards, relatórios e indicadores. |
| `resource/destino` | Arquivos exportados | Clientes / Setores | Entrega de arquivos finais. |
| `metadata` | Controle operacional | Engenharia de Dados | Logs, DDLs, checkpoints e auditoria. |
| `tmp` | Dados temporários | Pipelines | Processamento intermediário. |

---

## 10. Observações finais

Esta arquitetura serve como modelo inicial para organização de dados em projetos analíticos. A estrutura pode ser expandida conforme a maturidade do projeto, incluindo:

- catálogo de dados;
- controle de linhagem;
- validações de qualidade;
- políticas de retenção;
- controle de acesso por camada;
- versionamento de schemas;
- automação de cargas incrementais;
- monitoramento de pipelines.
